
# 00 The Geospatial Python Ecosystem: How It All Fits Together

This is the first notebook in a curriculum built specifically to get you from "confused by the docs" to
"able to build the geospatial features of the land banking platform." Every later notebook maps directly
to a real feature in that product's build matrix (duplicate-sale detection, boundary verification, survey
digitization, infrastructure-impact scoring). Nothing here is generic filler.

## Why these libraries feel confusing at first

The GeoPandas documentation is confusing to newcomers for one specific reason: **GeoPandas isn't one
library, it's a coordinating layer on top of several independent libraries, each solving one piece of the
"working with geographic data in Python" problem.** The docs assume you already know which piece is
which. This notebook fixes that before you touch any code.

## The mental model: five layers

Think of the ecosystem as five layers, each one built on the layer below it:

| Layer | Library | What it actually does | Do you call it directly? |
|---|---|---|---|
| 1. Geometry primitives | **Shapely** | Defines what a Point, LineString, and Polygon *are*, and the math for how they relate (intersects, contains, distance, buffer) | Yes, constantly this is the foundation everything else sits on |
| 2. Coordinate systems | **PyProj** | Converts coordinates between different reference systems (e.g. GPS lat/lon vs. a metric projection you need for accurate area/distance math) | Occasionally directly, but mostly through GeoPandas' `.crs` and `.to_crs()` |
| 3. File I/O | **Fiona** (and increasingly **Pyogrio**) | Reads and writes geographic file formats (Shapefile, GeoJSON, GeoPackage) | Almost never directly GeoPandas calls it for you when you use `gpd.read_file()` |
| 4. The table + geometry combo | **GeoPandas** | A pandas DataFrame where one column holds Shapely geometries instead of numbers/strings lets you filter, join, and analyze spatial data the same way you'd analyze a spreadsheet | Yes, constantly this is your main daily tool |
| 5. Spatial statistics & analysis | **PySAL** (and its sub-packages **libpysal**, **esda**) | Higher-level spatial statistics: spatial autocorrelation, clustering, spatial regression answers questions like "is high land value clustered geographically, or random?" | Yes, for the valuation-modeling and infrastructure-impact work specifically |

Three more libraries sit alongside this stack, each solving one specific, separate problem:

| Library | What it solves | Where you'll use it in this product |
|---|---|---|
| **Rtree** | Fast spatial lookups ("which of these 50,000 parcels are near this point?") without checking every single one | Under the hood of GeoPandas' `.sindex` you rarely call it directly, but it's *why* duplicate-sale detection stays fast at scale |
| **Geopy** | Geocoding turning an address or place name into coordinates, and vice versa | Converting a written address on a land document into a lat/lon point you can actually plot and query |
| **Psycopg2 + SQLAlchemy** | Talking to a PostgreSQL/PostGIS database from Python | Reading and writing your parcel/geometry data to the same Postgres database your Node.js backend already uses |

Two more names you'll see mentioned, worth knowing about even though you won't build lessons around them:

- **Geodatasets** a small helper library that fetches well-known sample geographic datasets (useful for practice, not for your real product data).
- **Descartes** an older library for plotting Shapely geometries with Matplotlib. **It's effectively deprecated** GeoPandas has had this plotting capability built in natively for years (`.plot()` on a GeoDataFrame just works). If you see a tutorial `import descartes`, it's outdated; you don't need it, and this curriculum won't use it.

## How this maps to the actual land banking features

| Feature from the product build matrix | Libraries you'll actually use |
|---|---|
| Duplicate-sale detection | Shapely + GeoPandas (+ Rtree under the hood) for polygon overlap checks; Psycopg2/SQLAlchemy to persist and query against PostGIS |
| Survey digitization (paper → georeferenced boundaries) | PyProj for coordinate system/projection handling |
| Infrastructure-impact forecasting | Geopy to geocode infrastructure locations; PySAL/esda for spatial clustering and proximity analysis |
| Valuation modeling | PySAL/esda for spatial autocorrelation as a feature input to your prediction model |

## A note on how this curriculum is organized

Each notebook after this one is self-contained, with working code you can run cell by cell, a short
conceptual explanation before each new idea (not after you should know *why* before you see *how*),
and exercises at the end with room to write your own answer before checking the solution.

Work through them in order 01 through 07 build on each other, and notebook 08 is the capstone,
a miniature but real version of the land banking platform's core duplicate-sale and verification engine.



## Setup check

Run the cell below. If it prints version numbers with no errors, your environment is ready for the
rest of this curriculum.


In [1]:

import geopandas, shapely, fiona, pyproj, geopy, rtree, libpysal, esda
import matplotlib
import mapclassify

print("geopandas:", geopandas.__version__)
print("shapely:  ", shapely.__version__)
print("fiona:    ", fiona.__version__)
print("pyproj:   ", pyproj.__version__)
print("geopy:    ", geopy.__version__)
print("rtree:    ", rtree.__version__)
print("libpysal: ", libpysal.__version__)
print("esda:     ", esda.__version__)
print()
print("Environment ready.")


geopandas: 1.1.4
shapely:   2.1.2
fiona:     1.10.1
pyproj:    3.7.2
geopy:     2.5.0
rtree:     1.4.1
libpysal:  4.15.0
esda:      2.10.0

Environment ready.



## If something failed to import

Install the full stack with:

```bash
pip install geopandas shapely fiona pyproj geopy rtree pysal esda libpysal splot \
            psycopg2-binary sqlalchemy matplotlib geodatasets mapclassify
```

On some systems, `fiona`, `rtree`, and `psycopg2` need system-level libraries (GDAL, libspatialindex,
libpq) installed *before* the `pip install` will succeed. If you hit a build error mentioning one of
those, search the error message plus your OS name the fix is almost always a one-line system package
install (e.g. `apt install libgdal-dev` on Ubuntu/Debian, `brew install gdal` on macOS) before retrying
the `pip install`.

## Next: `01_shapely_fundamentals.ipynb`

Start there everything else in this stack is built on the geometry objects Shapely defines.
